# 99_correct_place_names

## Objectif
Identifier et corriger manuellement les points GPX dont le reverse geocoding - suite au lancement du script - a retourné "Lieu inconnu" (0.2% des points, soit ~30 sur 14 765).

## Workflow
1. **Cellule 1** — Extraction des points problématiques → `data_raw/a_corriger.csv`
2. **Manuel** — Ouvrir `a_corriger.csv` dans Excel, identification chaque lieu via
   [Google Maps](https://maps.google.com) ou [OpenStreetMap](https://www.openstreetmap.org)
   en copiant les colonnes `lat` / `lon`, remplir la colonne `place_name`, enregistrer.
3. **Cellule 2** — Réinjection des corrections dans `data_raw/profile_srtm.csv`

## Fichiers concernés
- **Entrée** : `data_raw/profile_srtm.csv`
- **Intermédiaire** : `data_raw/a_corriger.csv` ← édition manuelle
- **Sortie** : `data_raw/profile_srtm.csv` (mis à jour in-place)


In [4]:
import pandas as pd

df = pd.read_csv("C:/Users/cello/Desktop/Via_Podiensis_geospatial/src/etl/data_raw/profile_srtm.csv")

# Identifier les blocs de "Lieu inconnu" et garder 1 seul représentant par bloc
# (le premier point de chaque séquence continue)
mask = df["place_name"] == "Lieu inconnu"
# Un nouveau bloc commence quand le point précédent était valide
nouveau_bloc = mask & (~mask.shift(1, fill_value=False))

jalons_inconnus = df[nouveau_bloc][["lat", "lon", "distance_m", "place_name"]].copy()
jalons_inconnus.index.name = "index_original"

print(f"Total points 'Lieu inconnu' : {mask.sum()}")
print(f"Blocs distincts à corriger  : {len(jalons_inconnus)}")
print()
print(jalons_inconnus.to_string())

jalons_inconnus.to_csv("C:/Users/cello/Desktop/Via_Podiensis_geospatial/data_raw/a_corriger.csv", index=True)
print("\nFichier a_corriger.csv généré.")

Total points 'Lieu inconnu' : 30
Blocs distincts à corriger  : 1

                     lat      lon   distance_m    place_name
index_original                                              
1322            44.84238  3.49701  59064.77019  Lieu inconnu

Fichier a_corriger.csv généré.


Crrection faite dans a_corriger.csv dans Excel : colonnes lat, lon, distance_m. Pour chaque ligne, verification lat,lon dans Google Maps ou OpenStreetMap pour identifier le lieu, puis réinjestion du nom de la localité dans la colonne place_name.

In [7]:
# Réinjection la correction dans le CSV principal

import pandas as pd

df = pd.read_csv("C:/Users/cello/Desktop/Via_Podiensis_geospatial/src/etl/data_raw/profile_srtm.csv")
corrections = pd.read_csv("C:/Users/cello/Desktop/Via_Podiensis_geospatial/data_raw/a_corriger.csv", index_col=0)

for idx, row in corrections.iterrows():
    # Trouver le début du bloc et propager jusqu'au prochain nom valide
    nom_corrige = row["place_name"]
    # Remplacer tous les "Lieu inconnu" consécutifs à partir de cet index
    i = idx
    while i < len(df) and df.loc[i, "place_name"] == "Lieu inconnu":
        df.loc[i, "place_name"] = nom_corrige
        i += 1

# Vérification
n_restants = (df["place_name"] == "Lieu inconnu").sum()
print(f"'Lieu inconnu' restants : {n_restants}")
print(df["place_name"].value_counts().head(10))

df.to_csv("C:/Users/cello/Desktop/Via_Podiensis_geospatial/src/etl/data_raw/profile_srtm.csv", index=False)
print("\nCSV mis à jour.")

'Lieu inconnu' restants : 0
place_name
Lectoure                   190
Cahors                     126
Peyre en Aubrac            121
Route de Saint-Jacques     120
Chemin de Saint-Jacques    117
Moissac                    113
Decazeville                106
Éauze                       98
Nasbinals                   97
Boudou                      95
Name: count, dtype: int64

CSV mis à jour.
